# Coverage-weighted demographics inside a radius — no GIS required

ZIP codes aren't points, and a radius almost always slices through them. The honest way
to ask *"how many people live within 10 miles of here?"* is to weight each ZIP by **how
much of it actually falls inside the circle** — a ZIP 30% inside should contribute 30% of
its population, not all of it and not none of it.

Normally that means loading boundary polygons into QGIS or PostGIS and doing the
intersection yourself. The [ZIP Codes API](https://www.zip-codes.com/api/) does it in one
REST call: `/radius` in **spatial mode** intersects your radius with actual ZIP/FSA boundary
polygons, returns each ZIP's `pct_inside`, and aggregates the Census ACS demographics
**coverage-weighted** — across any of the 14 available years (2011–2024).

**Runs as-is with the public demo key** (centered on demo ZIP `90210`). To center on *any*
US ZIP or a lat/lon coordinate, [grab a free key](https://www.zip-codes.com/api/signup) —
2,500 credits/day, no card.

## Population within 10 miles, 2011 vs 2024

One call asks for two ACS vintages at once (`acs_demographic` = current 2024, plus
`acs_demographic_2011`). The coverage-weighted aggregate comes back under `stats.acs`.

In [1]:
import requests

KEY = "zc_test_DEMOAPIKEY000000000000"  # demo key: works for demo ZIP centers (90210, 10001, ...)

resp = requests.get(
    "https://api.zip-codes.com/v2/radius",
    params={"code": "90210", "max": 10, "mode": "spatial",
            "include": "acs_demographic,acs_demographic_2011"},
    headers={"X-Api-Key": KEY}, timeout=60).json()
r = resp["results"][0]
acs = r["stats"]["acs"]
pop = {v["year"]: v["demographic"]["sex_and_age"]["total_population"]["est"]
       for v in acs.values()}
y0, y1 = min(pop), max(pop)
print(f"ZIPs intersecting the 10-mile radius of 90210: {r['summary']['total_count']}")
print("Population, coverage-weighted by actual overlap:")
print(f"  {y0}: {pop[y0]:,}")
print(f"  {y1}: {pop[y1]:,}")
print(f"  change {y0}-{y1}: {pop[y1]-pop[y0]:+,} ({(pop[y1]/pop[y0]-1)*100:+.1f}%)")

ZIPs intersecting the 10-mile radius of 90210: 154
Population, coverage-weighted by actual overlap:
  2011: 2,524,599
  2024: 2,555,579
  change 2011-2024: +30,980 (+1.2%)


## How the weighting works

The API reports its own aggregation method, and every match carries a `pct_inside`. Notice
how a ZIP barely clipped by the radius (90013, 0.1% inside) contributes almost nothing,
while one fully enclosed (90210, 100%) contributes in full:

In [2]:
note = list(acs.values())[0]["aggregation_note"]
print("How the aggregate is computed (reported by the API itself):")
print(" ", note)
print("\nHow much of each ZIP falls inside the radius (a few examples):")
for code in ["90210", "90272", "90065", "90013"]:
    m = next(x for x in r["matches"] if x["code"] == code)
    print(f"  {m['code']}  {m['city']:<16} {m['pct_inside']:>5}% inside   ({m['distance_miles']} mi out)")

How the aggregate is computed (reported by the API itself):
  Values are area-weighted aggregates across all US ZIPs intersecting the radius. Counts use sum(value * pct_inside). Medians/means use population-weighted average. Rates are recomputed from aggregated numerator/denominator. Percentages are recomputed from aggregated sums. No MOE in summary.

How much of each ZIP falls inside the radius (a few examples):
  90210  Beverly Hills      100% inside   (0 mi out)
  90272  Pacific Palisades  98.9% inside   (8.13 mi out)
  90065  Los Angeles       10.7% inside   (10.56 mi out)
  90013  Los Angeles        0.1% inside   (10.87 mi out)


## Adapt this to your question

The demo centers on Beverly Hills because the public demo key is limited to demo ZIPs. With
a [free key](https://www.zip-codes.com/api/signup) you can change the center to **any US ZIP
or a `lat,lon` coordinate** — a hospital, a fab, a toxic-release site, a proposed store — and
swap the profile/year to fit the question:

- **Profiles**: `acs_demographic` (DP05), `acs_social` (DP02), `acs_economic` (DP03),
  `acs_housing` (DP04) — append `_YYYY` for any year 2011–2024.
- **Center**: `code=44256` or `code=41.06,-81.52`; **radius**: `max=10` (miles; spatial mode up to 250).
- **Mode**: `mode=spatial` for boundary-accurate coverage weighting; `mode=centroid` for a faster point approximation.

**Honest caveats.** Aggregates cover the ZIPs that have ACS data (ZCTAs); PO-box-only and
sliver ZIPs without a ZCTA are reported in `coverage_pct` and excluded — in this example,
99 of the 154 intersecting ZIPs carry ACS data. ACS values are 5-Year survey estimates with
margins of error; the per-ZIP `/zip` endpoint returns MOE for each field (the radius summary omits it).

### Credits & citation

- Free tier: 2,500 credits/day, no card ([signup](https://www.zip-codes.com/api/signup)).
  Researchers, journalists, students, and nonprofits can request extended credits in exchange
  for a data acknowledgement — [details](https://www.zip-codes.com/api/research-credits) or
  email <jharris@zip-codes.com>.
- Docs: <https://www.zip-codes.com/api/docs> · method reference: <https://api.zip-codes.com/llms-full.txt>

Suggested acknowledgement:

> Demographic data: US Census Bureau ACS 5-Year Estimates, accessed via ZIP Codes API
> (Zip-Codes.com), https://www.zip-codes.com/api/